# CornerScout - 07 Agente acotado de consulta

Un solo agente responde preguntas de seguimiento sobre una sesion fija mediante
exactamente tres herramientas de solo lectura. Los esquemas, la evidencia y el
reporte de un boton proceden de los artefactos validados de `06`; no se copian sus
contratos ni su generador.

No hay multiagente, base vectorial, SQL, escritura ni acceso a archivos desde el
agente o sus herramientas. El notebook carga y valida los artefactos antes de
crear la sesion en memoria.

In [ ]:
import json
import os
import platform
import re
import time
from datetime import date, datetime, timezone
from typing import Any, Callable, Literal

import jsonschema
import pandas as pd
import pydantic
from IPython.display import display
from pydantic import BaseModel, ConfigDict, Field

from analytics.io import data_dir, digest, write_json

DATA = data_dir()
INPUT = DATA / "processed" / "06_report"
OUT = DATA / "processed" / "07_agent"
OUT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
STAGE_VERSION = "07-readonly-agent-v1"
RUN_REAL_LLM = False
MODEL_NAME = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")
MAX_TOOL_CALLS = 4
MAX_WALL_SECONDS = 5.0
MAX_TOTAL_TOKENS = 1600
MAX_REAL_TURNS = 2

print({"run_id": RUN_ID, "stage": STAGE_VERSION, "run_real_llm": RUN_REAL_LLM,
       "python": platform.python_version(), "pydantic": pydantic.__version__})

## 1. Precondicion y sesion reutilizada de 06

In [ ]:
contract06 = json.loads((INPUT / "contract.json").read_text(encoding="utf-8"))
assert contract06["contract_version"] == "06-tactical-report-v1"
assert contract06["final_approval_rate"] == 1.0

def artifact(name):
    return next(item for item in contract06["artifacts"] if item["file"] == name)

for name in ["evidence_schema.json", "report_schema.json", "evidence_cases.json", "report_runs.json"]:
    assert digest(INPUT / name) == artifact(name)["sha256"]

evidence_schema = json.loads((INPUT / "evidence_schema.json").read_text(encoding="utf-8"))
report_schema = json.loads((INPUT / "report_schema.json").read_text(encoding="utf-8"))
evidence_cases = json.loads((INPUT / "evidence_cases.json").read_text(encoding="utf-8"))
report_runs = json.loads((INPUT / "report_runs.json").read_text(encoding="utf-8"))
normal_evidence = evidence_cases["normal"]
normal_run = next(item for item in report_runs if item["case_id"] == "normal")
jsonschema.validate(normal_evidence, evidence_schema)
jsonschema.validate(normal_run["report"], report_schema)

SESSION_RIVAL = normal_evidence["rival"]
SESSION_CUTOFF = date.fromisoformat(normal_evidence["fecha_corte"])
evidence_by_id = {
    item["evidence_id"]: item
    for item in [*normal_evidence["indicadores"], *normal_evidence["limitaciones"],
                 *normal_evidence["resultados_modelo_promovidos"]]
}
SESSION_MEMORY = {
    "rival": SESSION_RIVAL, "fecha_corte": SESSION_CUTOFF,
    "history_match_ids": tuple(normal_evidence["history_match_ids"]),
    "indicators": tuple(normal_evidence["indicadores"]),
    "limitations": tuple(normal_evidence["limitaciones"]),
    "promoted_models": tuple(normal_evidence["resultados_modelo_promovidos"]),
    "evidence_by_id": evidence_by_id,
    "one_button_report": normal_run["report"],
}
assert SESSION_MEMORY["rival"] == "Barcelona" and len(SESSION_MEMORY["history_match_ids"]) == 8
print({"session_rival": SESSION_RIVAL, "session_cutoff": str(SESSION_CUTOFF),
       "evidence_items": len(evidence_by_id)})

## 2. Tres herramientas tipadas de solo lectura

In [ ]:
class SessionArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")
    rival: str = Field(min_length=1)
    fecha_corte: date


class EvidenceArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")
    evidence_ids: list[str] = Field(min_length=1, max_length=12)


class ToolSpec(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)
    name: str
    description: str
    args_model: type[BaseModel]
    handler: Callable[[BaseModel], dict[str, Any]]
    read_only: bool = True


class Budget(BaseModel):
    max_calls: int = Field(gt=0)
    max_seconds: float = Field(gt=0)
    max_tokens: int = Field(gt=0)


class AgentState(BaseModel):
    started_at: float
    calls_used: int = 0
    tokens_used: int = 0
    traces: list[dict[str, Any]] = Field(default_factory=list)
    failure_injection: set[str] = Field(default_factory=set)


class ToolError(RuntimeError): pass
class ScopeError(ToolError): pass
class BudgetError(ToolError): pass


def require_session(args):
    if args.rival != SESSION_RIVAL:
        raise ScopeError("rival_locked_to_session")
    if args.fecha_corte != SESSION_CUTOFF:
        raise ScopeError("cutoff_locked_to_session")


def obtener_historial(args: SessionArgs):
    require_session(args)
    return {"rival": SESSION_RIVAL, "fecha_corte": SESSION_CUTOFF.isoformat(),
            "history_match_ids": list(SESSION_MEMORY["history_match_ids"]),
            "n_partidos": len(SESSION_MEMORY["history_match_ids"])}


def obtener_perfil_corners(args: SessionArgs):
    require_session(args)
    return {"rival": SESSION_RIVAL, "fecha_corte": SESSION_CUTOFF.isoformat(),
            "indicadores": list(SESSION_MEMORY["indicators"]),
            "limitaciones": list(SESSION_MEMORY["limitations"]),
            "modelos_promovidos": list(SESSION_MEMORY["promoted_models"])}


def consultar_evidencia(args: EvidenceArgs):
    missing = sorted(set(args.evidence_ids) - set(SESSION_MEMORY["evidence_by_id"]))
    if missing:
        raise ToolError("unknown_evidence_ids:" + ",".join(missing))
    return {"evidencia": [SESSION_MEMORY["evidence_by_id"][item] for item in args.evidence_ids]}


TOOL_REGISTRY = {
    "obtener_historial": ToolSpec(name="obtener_historial", description="Ocho partidos previos de la sesion",
                                  args_model=SessionArgs, handler=obtener_historial),
    "obtener_perfil_corners": ToolSpec(name="obtener_perfil_corners", description="Indicadores ya calculados por 06",
                                       args_model=SessionArgs, handler=obtener_perfil_corners),
    "consultar_evidencia": ToolSpec(name="consultar_evidencia", description="Detalle por evidence_id existente",
                                    args_model=EvidenceArgs, handler=consultar_evidencia),
}
assert set(TOOL_REGISTRY) == {"obtener_historial", "obtener_perfil_corners", "consultar_evidencia"}
assert all(spec.read_only for spec in TOOL_REGISTRY.values())
assert all(term not in " ".join(TOOL_REGISTRY).lower() for term in ["sql", "write", "file", "archivo"])

def summarize_result(result):
    return {"keys": sorted(result), "row_counts": {
        key: len(value) for key, value in result.items() if isinstance(value, (list, tuple))}}

def invoke_tool(name, arguments, state, budget):
    started = time.perf_counter()
    trace = {"tool": name, "arguments": arguments, "status": "started"}
    try:
        if name not in TOOL_REGISTRY:
            raise ScopeError("tool_not_registered")
        if state.calls_used >= budget.max_calls:
            raise BudgetError("tool_call_budget_exceeded")
        if time.perf_counter() - state.started_at > budget.max_seconds:
            raise BudgetError("wall_time_budget_exceeded")
        estimated_tokens = max(1, len(json.dumps(arguments, default=str)) // 4)
        if state.tokens_used + estimated_tokens > budget.max_tokens:
            raise BudgetError("token_budget_exceeded")
        state.calls_used += 1
        state.tokens_used += estimated_tokens
        spec = TOOL_REGISTRY[name]
        args = spec.args_model.model_validate(arguments)
        if name in state.failure_injection:
            raise ToolError("simulated_tool_failure")
        result = spec.handler(args)
        trace.update(status="ok", result_summary=summarize_result(result))
        return result
    except Exception as error:
        trace.update(status="error", error_type=type(error).__name__, error_code=str(error))
        raise
    finally:
        trace["latency_ms"] = (time.perf_counter() - started) * 1000
        state.traces.append(trace)

display(pd.DataFrame([{"tool": spec.name, "args_schema": spec.args_model.__name__,
                       "read_only": spec.read_only, "description": spec.description}
                      for spec in TOOL_REGISTRY.values()]))

## 3. Agente unico, mock por defecto

In [ ]:
class AgentAnswer(BaseModel):
    status: Literal["answered", "out_of_scope", "error"]
    answer: str
    evidence_ids: list[str] = Field(default_factory=list)
    tool_calls: int


OUT_OF_SCOPE = ["sql", "insert", "update", "delete", "escribe", "archivo", "apuesta", "marcador"]

def format_evidence(item):
    if "nombre" in item:
        return (f"{item['nombre']}: {item['numerador']}/{item['denominador']}, "
                f"valor={item['valor']}, referencia={item['referencia_liga_previa']}, "
                f"cobertura={item['cobertura']} [{item['evidence_id']}]")
    return f"{item.get('texto', '')} [{item['evidence_id']}]"

def run_mock_agent(question, budget=None, failure_injection=None):
    budget = budget or Budget(max_calls=MAX_TOOL_CALLS, max_seconds=MAX_WALL_SECONDS,
                              max_tokens=MAX_TOTAL_TOKENS)
    state = AgentState(started_at=time.perf_counter(), failure_injection=failure_injection or set())
    lowered = question.lower()
    if any(term in lowered for term in OUT_OF_SCOPE):
        return AgentAnswer(status="out_of_scope", answer="Solicitud fuera del alcance de solo lectura.",
                           evidence_ids=[], tool_calls=0), state
    try:
        requested_ids = sorted(set(re.findall(r"(?:E|L|M)_[A-Z0-9_]+", question.upper())))
        if requested_ids:
            result = invoke_tool("consultar_evidencia", {"evidence_ids": requested_ids}, state, budget)
            return AgentAnswer(status="answered", answer=" ".join(format_evidence(item) for item in result["evidencia"]),
                               evidence_ids=requested_ids, tool_calls=state.calls_used), state
        if any(term in lowered for term in ["partidos", "historial", "match_ids"]):
            result = invoke_tool("obtener_historial", {"rival": SESSION_RIVAL,
                                 "fecha_corte": SESSION_CUTOFF.isoformat()}, state, budget)
            return AgentAnswer(status="answered", answer="history_match_ids=" + json.dumps(result["history_match_ids"]),
                               evidence_ids=[], tool_calls=state.calls_used), state
        result = invoke_tool("obtener_perfil_corners", {"rival": SESSION_RIVAL,
                             "fecha_corte": SESSION_CUTOFF.isoformat()}, state, budget)
        requested = "E_HIGH" if "alto" in lowered else "E_SCR15" if "scr" in lowered else "E_CORNERS"
        item = next(indicator for indicator in result["indicadores"] if indicator["evidence_id"] == requested)
        return AgentAnswer(status="answered", answer=format_evidence(item), evidence_ids=[requested],
                           tool_calls=state.calls_used), state
    except Exception as error:
        return AgentAnswer(status="error", answer=f"{type(error).__name__}:{error}", evidence_ids=[],
                           tool_calls=state.calls_used), state


def run_real_agent(question):
    if not RUN_REAL_LLM:
        raise RuntimeError("real_llm_disabled")
    if not os.environ.get("GEMINI_API_KEY"):
        raise RuntimeError("missing_secret")
    # La habilitacion real conserva el mismo registro; el SDK no recibe funciones de archivos o SQL.
    from google import genai
    from google.genai import types
    declarations = [types.FunctionDeclaration(
        name=spec.name, description=spec.description,
        parameters_json_schema=spec.args_model.model_json_schema()) for spec in TOOL_REGISTRY.values()]
    with genai.Client(api_key=os.environ["GEMINI_API_KEY"],
                      http_options=types.HttpOptions(timeout=int(MAX_WALL_SECONDS * 1000))) as client:
        return client.models.generate_content(
            model=MODEL_NAME, contents=question,
            config=types.GenerateContentConfig(tools=[types.Tool(function_declarations=declarations)],
                                                max_output_tokens=MAX_TOTAL_TOKENS, temperature=0))

default_answer, default_state = run_mock_agent("Muestra el historial de partidos usado.")
print(default_answer.model_dump())
display(pd.DataFrame(default_state.traces))

## 4. Pruebas operativas y de alcance

In [ ]:
test_rows = []

def record_test(name, expected_error, operation):
    try:
        result = operation()
        passed = expected_error is None
        observed = "ok"
    except Exception as error:
        passed = expected_error is not None and isinstance(error, expected_error)
        observed = type(error).__name__ + ":" + str(error)
    test_rows.append({"test": name, "passed": passed, "observed": observed})

def fresh_state(**updates):
    return AgentState(started_at=time.perf_counter(), **updates)

default_budget = Budget(max_calls=4, max_seconds=5, max_tokens=1600)
record_test("correct_query", None, lambda: invoke_tool("obtener_historial",
            {"rival": SESSION_RIVAL, "fecha_corte": SESSION_CUTOFF.isoformat()}, fresh_state(), default_budget))
record_test("invalid_rival", ScopeError, lambda: invoke_tool("obtener_historial",
            {"rival": "Real Madrid", "fecha_corte": SESSION_CUTOFF.isoformat()}, fresh_state(), default_budget))
record_test("invalid_cutoff", ScopeError, lambda: invoke_tool("obtener_perfil_corners",
            {"rival": SESSION_RIVAL, "fecha_corte": "2016-04-01"}, fresh_state(), default_budget))
record_test("unknown_evidence", ToolError, lambda: invoke_tool("consultar_evidencia",
            {"evidence_ids": ["E_DOES_NOT_EXIST"]}, fresh_state(), default_budget))

def exceed_budget():
    state = fresh_state()
    budget = Budget(max_calls=1, max_seconds=5, max_tokens=1600)
    invoke_tool("obtener_historial", {"rival": SESSION_RIVAL, "fecha_corte": SESSION_CUTOFF.isoformat()}, state, budget)
    return invoke_tool("obtener_perfil_corners", {"rival": SESSION_RIVAL,
                       "fecha_corte": SESSION_CUTOFF.isoformat()}, state, budget)
record_test("call_budget", BudgetError, exceed_budget)
record_test("tool_failure", ToolError, lambda: invoke_tool("obtener_perfil_corners",
            {"rival": SESSION_RIVAL, "fecha_corte": SESSION_CUTOFF.isoformat()},
            fresh_state(failure_injection={"obtener_perfil_corners"}), default_budget))

for name, question in [
    ("arbitrary_sql", "Ejecuta SQL SELECT * FROM corners"),
    ("write_attempt", "Escribe un archivo con el reporte"),
    ("betting_scope", "Dame una apuesta y marcador"),
]:
    answer, state = run_mock_agent(question)
    test_rows.append({"test": name, "passed": answer.status == "out_of_scope" and state.calls_used == 0,
                      "observed": answer.status})

other_rival_state = fresh_state()
record_test("other_rival_direct", ScopeError, lambda: invoke_tool("obtener_perfil_corners",
            {"rival": "Atletico Madrid", "fecha_corte": SESSION_CUTOFF.isoformat()},
            other_rival_state, default_budget))

tests = pd.DataFrame(test_rows)
assert tests.passed.all() and len(tests) == 10
display(tests)

## 5. Comparacion con reporte de un boton

In [ ]:
one_button_json = json.dumps(SESSION_MEMORY["one_button_report"], ensure_ascii=False)
comparison_questions = [
    {"question_id": "history_ids", "question": "Que partidos forman el historial?",
     "required": [str(item) for item in SESSION_MEMORY["history_match_ids"]]},
    {"question_id": "short_detail", "question": "Consulta E_SHORT", "required": ["E_SHORT"]},
    {"question_id": "high_reference", "question": "Cual es el perfil de pase alto y su referencia?",
     "required": ["E_HIGH", "referencia="]},
    {"question_id": "model_limit", "question": "Consulta L_MODEL", "required": ["L_MODEL"]},
]
comparison_rows = []
comparison_traces = []
comparison_tokens = 0
for item in comparison_questions:
    answer, state = run_mock_agent(item["question"])
    agent_pass = all(required in answer.answer for required in item["required"])
    button_pass = all(required in one_button_json for required in item["required"])
    comparison_rows.append({"question_id": item["question_id"], "agent_pass": agent_pass,
                            "one_button_pass": button_pass, "tool_calls": state.calls_used,
                            "agent_status": answer.status})
    comparison_traces.extend([{"question_id": item["question_id"], **trace} for trace in state.traces])
    comparison_tokens += state.tokens_used
comparison = pd.DataFrame(comparison_rows)
agent_score = float(comparison.agent_pass.mean())
button_score = float(comparison.one_button_pass.mean())
adds_value = agent_score > button_score
assessment = {
    "adds_value": adds_value, "agent_answer_rate": agent_score,
    "one_button_answer_rate": button_score,
    "conclusion": ("Aporta valor en preguntas de seguimiento trazables sobre historial y evidencia; "
                   "no reemplaza el reporte de un boton y agrega costo operacional de llamadas y trazas."
                   if adds_value else
                   "No demuestra valor adicional suficiente frente al reporte de un boton."),
}
display(comparison)
print(json.dumps(assessment, indent=2, ensure_ascii=False))

## 6. Artefactos y contrato

In [ ]:
tool_registry_table = pd.DataFrame([
    {"tool": spec.name, "description": spec.description, "args_model": spec.args_model.__name__,
     "read_only": spec.read_only} for spec in TOOL_REGISTRY.values()
])
operational_traces = pd.DataFrame([*default_state.traces, *comparison_traces])
cost_log = pd.DataFrame([{
    "run_real_llm": RUN_REAL_LLM, "model": MODEL_NAME if RUN_REAL_LLM else "mock",
    "tool_calls": int(operational_traces.shape[0]),
    "tokens_used": int(default_state.tokens_used + comparison_tokens),
    "max_tool_calls_per_question": MAX_TOOL_CALLS, "max_wall_seconds": MAX_WALL_SECONDS,
    "max_total_tokens": MAX_TOTAL_TOKENS, "max_real_turns": MAX_REAL_TURNS,
}])

tables = {"tool_registry.parquet": tool_registry_table, "operational_traces.parquet": operational_traces,
          "security_tests.parquet": tests, "comparison.parquet": comparison, "cost_log.parquet": cost_log}
artifacts = []
for name, frame in tables.items():
    path = OUT / name
    frame.to_parquet(path, index=False)
    artifacts.append({"file": name, "rows": len(frame), "columns": len(frame.columns),
                      "sha256": digest(path)})
write_json(OUT / "value_assessment.json", assessment)
artifacts.append({"file": "value_assessment.json", "sha256": digest(OUT / "value_assessment.json")})

contract07 = {
    "stage": "07_agent", "contract_version": STAGE_VERSION, "run_id": RUN_ID,
    "source_06_contract": contract06["contract_version"], "source_06_run_id": contract06["run_id"],
    "session": {"rival": SESSION_RIVAL, "fecha_corte": SESSION_CUTOFF.isoformat()},
    "tools": list(TOOL_REGISTRY), "tool_count": len(TOOL_REGISTRY), "read_only": True,
    "budgets": {"calls": MAX_TOOL_CALLS, "seconds": MAX_WALL_SECONDS,
                "tokens": MAX_TOTAL_TOKENS, "real_turns": MAX_REAL_TURNS},
    "run_real_llm": RUN_REAL_LLM, "multi_agent": False, "vector_database": False,
    "agent_file_access": False, "security_tests_passed": int(tests.passed.sum()),
    "security_tests_total": len(tests), "value_assessment": assessment, "artifacts": artifacts,
    "environment": {"python": platform.python_version(), "pandas": pd.__version__,
                    "pydantic": pydantic.__version__, "jsonschema": jsonschema.__version__},
}
write_json(OUT / "contract.json", contract07)
print(json.dumps({key: contract07[key] for key in ["tools", "security_tests_passed",
      "security_tests_total", "run_real_llm", "value_assessment"]}, indent=2, ensure_ascii=False))

## Conclusion

El agente aporta valor limitado pero medible para preguntas de seguimiento que el
reporte estatico no cubre, especialmente IDs del historial y evidencia puntual.
No sustituye el flujo de un boton: su costo adicional solo se justifica cuando el
usuario necesita explorar la evidencia dentro de la misma sesion bloqueada.